
# Assignment 04 · Miền MNIST · Notebook 02: CNN sâu với PyTorch và Keras, đối chiếu ba cách cài đặt

**Học phần:** Phát triển các Hệ thống Thông minh, Học viện Công nghệ Bưu chính Viễn thông

**Sinh viên:** Nguyễn Duy Nghĩa &nbsp;·&nbsp; **Mã sinh viên:** B23DCCN600 &nbsp;·&nbsp; **Lớp:** D23CTPM01

**Giảng viên hướng dẫn:** PGS.TS Trần Đình Quế

**Học kỳ:** Học kỳ 1 năm học 2026 &ndash; 2027

---

## Mục tiêu notebook

Notebook 01 đã chứng minh rằng phép lan truyền ngược viết tay là đúng và một mạng tích chập hai
tầng bằng NumPy thuần đạt $98{,}23\%$ trên tập kiểm thử. Notebook 02 chuyển sang hai thư viện học
sâu công nghiệp và trả lời ba câu hỏi:

1. Khi được trang bị Batch Normalization, Dropout và toàn bộ 48 000 ảnh huấn luyện, mạng tích
   chập sâu đạt tới đâu trên MNIST?
2. PyTorch và Keras, với cùng kiến trúc, cùng bộ tối ưu, cùng seed và cùng phép chuẩn hóa, có
   cho kết quả tương đương nhau không?
3. Ba cách cài đặt (NumPy thuần, PyTorch, Keras) chênh nhau bao nhiêu về độ chính xác, thời gian
   huấn luyện và số dòng mã cần viết?

Notebook cũng chịu trách nhiệm tạo ra hai hiện vật dùng cho notebook `mlp_vs_cnn`:
tệp trọng số `models/mnist_cnn_pytorch.pt`, mô-đun định nghĩa lớp `models/mnist_cnn_def.py`, và
hằng số tiền xử lý `models/mnist_preproc.json`. Mô hình PyTorch bắt buộc có phương thức
`extract_features(x)` trả về véc-tơ ẩn 128 chiều để phân tích PCA không gian ẩn.

Năm hình bắt buộc notebook này sinh ra: `fig_mnist_framework_curves.png`,
`fig_mnist_framework_confusion.png`, `fig_mnist_3way_benchmark.png`,
`fig_mnist_per_class_accuracy.png`, `fig_mnist_high_conf_errors.png`, cùng với tệp tổng hợp
`reports/metrics_mnist.json`.


## 1. Nhập thư viện và cấu hình seed

Ba nguồn ngẫu nhiên phải được cố định độc lập: `np.random.seed` cho NumPy và scikit-learn,
`torch.manual_seed` cho PyTorch, `keras.utils.set_random_seed` cho Keras. Nếu bỏ sót bất kỳ
nguồn nào, hai lần chạy sẽ cho kết quả khác nhau và phép so sánh mất ý nghĩa.

In [1]:

import os, json, time, copy, sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (precision_recall_fscore_support, confusion_matrix,
                             classification_report)

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')
import tensorflow as tf
import keras
from keras import layers

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
keras.utils.set_random_seed(RANDOM_SEED)
torch.use_deterministic_algorithms(False)

plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = ['Segoe UI', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH  = '../data/mnist.npz'
FIG_DIR    = '../reports/figures'
REP_DIR    = '../reports'
MODEL_DIR  = '../models'
for d in (FIG_DIR, REP_DIR, MODEL_DIR):
    os.makedirs(d, exist_ok=True)

EPOCHS_FW  = 10
BATCH_FW   = 128
DEVICE     = torch.device('cpu')

print('NumPy      :', np.__version__)
print('PyTorch    :', torch.__version__, '| CUDA khả dụng:', torch.cuda.is_available())
print('TensorFlow :', tf.__version__)
print('Keras      :', keras.__version__)
print('Thiết bị   :', DEVICE, f'| số luồng CPU PyTorch = {torch.get_num_threads()}')
print(f'Cấu hình huấn luyện framework: {EPOCHS_FW} epoch, batch {BATCH_FW}')

NumPy      : 2.4.0
PyTorch    : 2.9.1+cpu | CUDA khả dụng: False
TensorFlow : 2.21.0
Keras      : 3.15.1
Thiết bị   : cpu | số luồng CPU PyTorch = 16
Cấu hình huấn luyện framework: 10 epoch, batch 128



## 2. Nạp dữ liệu, chia tập, chuẩn hóa và lưu hằng số tiền xử lý

Quy trình lặp lại **chính xác** những gì notebook 01 đã làm, với một khác biệt duy nhất: hai
framework dùng **toàn bộ** 48 000 ảnh nhánh train và 12 000 ảnh validation, không lấy tập con.
Lý do là PyTorch và Keras chạy nhanh hơn NumPy thuần khoảng một bậc độ lớn nhờ nhân GEMM đa
luồng, nên toàn bộ dữ liệu vẫn nằm trong ngân sách thời gian.

Hai hằng số $\mu$ và $\sigma$ được tính lại từ cùng một nhánh train với cùng `random_state=42`,
do đó trùng khít với giá trị của notebook 01. Chúng được ghi ra `models/mnist_preproc.json`
để notebook `mlp_vs_cnn` nạp đúng phép biến đổi khi tải lại mô hình PyTorch đã huấn luyện. Nếu
không lưu, việc suy luận sau này với phép chuẩn hóa khác sẽ cho kết quả sai lệch mà không có bất
kỳ cảnh báo nào.

In [2]:

_d = np.load(DATA_PATH)
x_train_raw, y_train_raw = _d['x_train'], _d['y_train'].astype(np.int64)
x_test_raw,  y_test_raw  = _d['x_test'],  _d['y_test'].astype(np.int64)

idx_all = np.arange(len(x_train_raw))
idx_tr, idx_va = train_test_split(idx_all, test_size=0.2,
                                  stratify=y_train_raw, random_state=RANDOM_SEED)

MEAN = float((x_train_raw[idx_tr].astype(np.float32) / 255.0).mean())
STD  = float((x_train_raw[idx_tr].astype(np.float32) / 255.0).std())

def preprocess(x_u8):
    x = x_u8.astype(np.float32) / 255.0
    return ((x - MEAN) / STD)[:, None, :, :]      # NCHW

X_tr, y_tr = preprocess(x_train_raw[idx_tr]), y_train_raw[idx_tr]
X_va, y_va = preprocess(x_train_raw[idx_va]), y_train_raw[idx_va]
X_te, y_te = preprocess(x_test_raw),          y_test_raw

print(f'MEAN = {MEAN:.10f}   STD = {STD:.10f}')
print(f'Train      : {X_tr.shape}  phân phối lớp {np.bincount(y_tr, minlength=10).tolist()}')
print(f'Validation : {X_va.shape}  phân phối lớp {np.bincount(y_va, minlength=10).tolist()}')
print(f'Test       : {X_te.shape}  phân phối lớp {np.bincount(y_te, minlength=10).tolist()}')
print(f'Khoảng giá trị sau chuẩn hóa: [{X_tr.min():.4f}, {X_tr.max():.4f}]')

preproc = {
    'description': 'Hằng số chuẩn hóa MNIST, học từ 48000 ảnh nhánh train '
                   '(train_test_split test_size=0.2, stratify=y, random_state=42).',
    'mean': MEAN, 'std': STD, 'scale': 255.0,
    'formula': 'x_norm = (x_uint8 / 255.0 - mean) / std',
    'input_layout': 'NCHW (N, 1, 28, 28) cho PyTorch',
    'n_train': int(len(idx_tr)), 'n_val': int(len(idx_va)), 'n_test': int(len(X_te)),
    'random_seed': RANDOM_SEED,
}
with open(os.path.join(MODEL_DIR, 'mnist_preproc.json'), 'w', encoding='utf-8') as f:
    json.dump(preproc, f, ensure_ascii=False, indent=2)
print()
print('Đã lưu', os.path.join(MODEL_DIR, 'mnist_preproc.json'))

MEAN = 0.1307886839   STD = 0.3082403541
Train      : (48000, 1, 28, 28)  phân phối lớp [4738, 5394, 4766, 4905, 4674, 4337, 4734, 5012, 4681, 4759]
Validation : (12000, 1, 28, 28)  phân phối lớp [1185, 1348, 1192, 1226, 1168, 1084, 1184, 1253, 1170, 1190]
Test       : (10000, 1, 28, 28)  phân phối lớp [980, 1135, 1032, 1010, 982, 892, 958, 1028, 974, 1009]
Khoảng giá trị sau chuẩn hóa: [-0.4243, 2.8199]

Đã lưu ../models\mnist_preproc.json



## 3. Cơ sở toán học của hai thành phần mới: Batch Normalization và Dropout

Kiến trúc ở notebook 01 chỉ có tích chập, ReLU, pooling và tầng kết nối đầy đủ. Kiến trúc của
notebook này bổ sung hai thành phần chính quy hóa mà mục 4 hợp đồng quy định.

### 3.1 Batch Normalization

Với một lô $\mathcal{B}$ gồm $m$ mẫu, BatchNorm chuẩn hóa **theo từng kênh** rồi biến đổi affine
bằng hai tham số học được $\gamma$ và $\beta$:

$$
\mu_{\mathcal{B}} = \frac{1}{m}\sum_{i=1}^{m} x_i,
\qquad
\sigma^2_{\mathcal{B}} = \frac{1}{m}\sum_{i=1}^{m} (x_i - \mu_{\mathcal{B}})^2,
$$

$$
\hat{x}_i = \frac{x_i - \mu_{\mathcal{B}}}{\sqrt{\sigma^2_{\mathcal{B}} + \epsilon}},
\qquad
y_i = \gamma \hat{x}_i + \beta .
$$

Với đầu vào tích chập $(N, C, H, W)$, thống kê được lấy trên ba trục $N$, $H$, $W$ nên mỗi kênh
có riêng một cặp $(\gamma, \beta)$, tổng cộng $2C$ tham số học được cộng $2C$ bộ đệm trung bình
trượt không học được.

Vai trò của BatchNorm gồm ba mặt. Thứ nhất, nó giữ phân phối đầu vào của mỗi tầng ổn định qua
các bước lặp, giảm hiện tượng dịch chuyển hiệp biến nội bộ, nhờ đó cho phép dùng learning rate
lớn hơn. Thứ hai, nó làm mặt mất mát trơn hơn nên gradient đáng tin cậy hơn trên quãng dài. Thứ
ba, vì thống kê được ước lượng trên từng lô ngẫu nhiên, nó bơm vào mạng một lượng nhiễu nhỏ và
do đó có tác dụng chính quy hóa nhẹ.

Điểm khác biệt quan trọng giữa hai chế độ: khi huấn luyện, BatchNorm dùng thống kê của lô hiện
tại; khi suy luận, nó dùng trung bình trượt $\mathbb{E}[x]$ và $\mathrm{Var}[x]$ tích lũy trong
quá trình huấn luyện. Quên chuyển mô hình sang chế độ đánh giá là lỗi kinh điển làm độ chính xác
tụt vài điểm phần trăm mà không có thông báo lỗi nào. Trong PyTorch, chuyển chế độ bằng
`model.eval()`; Keras tự xử lý qua cờ `training`.

### 3.2 Dropout

Trong pha huấn luyện, Dropout với xác suất $p$ nhân đầu ra với một mặt nạ Bernoulli rồi chia lại
để giữ kỳ vọng không đổi (kiểu inverted dropout):

$$
r_j \sim \mathrm{Bernoulli}(1-p), \qquad
\tilde{h}_j = \frac{r_j}{1-p}\, h_j ,
\qquad \mathbb{E}\!\left[\tilde{h}_j\right] = h_j .
$$

Ở pha suy luận, Dropout trở thành phép đồng nhất. Cách chia lại cho $1-p$ ngay khi huấn luyện
giúp không phải nhân tỉ lệ ở pha suy luận, đó là lý do nó được mọi thư viện hiện đại lựa chọn.

Dropout buộc mạng không phụ thuộc vào một nơ-ron đơn lẻ nào, tương đương với việc lấy trung bình
ngầm định trên một họ rất lớn các mạng con. Báo cáo dùng $p = 0{,}25$ sau mỗi khối tích chập và
$p = 0{,}5$ sau tầng ẩn 128 chiều, đúng theo cấu hình thông dụng: tầng kết nối đầy đủ có nhiều
tham số nhất nên cần chính quy hóa mạnh nhất.

### 3.3 Kiến trúc hai khối phân cấp

```
Đầu vào (1, 28, 28)
  Khối 1: Conv2D(32, 3x3, same) -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout(0.25)   => (32, 14, 14)
  Khối 2: Conv2D(64, 3x3, same) -> BatchNorm -> ReLU -> MaxPool(2) -> Dropout(0.25)   => (64,  7,  7)
  Flatten  => 3136
  Dense(128) -> ReLU -> Dropout(0.5)    <-- véc-tơ ẩn 128 chiều, đầu ra của extract_features
  Dense(10)  -> Softmax
```

Số kênh tăng dần $32 \to 64$ trong khi kích thước không gian giảm $28 \to 14 \to 7$. Đây là
nguyên tắc thiết kế phân cấp: các tầng nông nhìn thấy vùng nhỏ và học đặc trưng cục bộ như biên
và góc nên cần ít kênh, các tầng sâu có trường tiếp nhận lớn hơn và học tổ hợp phức tạp hơn nên
cần nhiều kênh. Tích $C \times H \times W$ nhờ đó gần như không đổi giữa hai khối
($32 \cdot 14 \cdot 14 = 6\,272$ so với $64 \cdot 7 \cdot 7 = 3\,136$), giữ chi phí tính toán
cân đối.


## 4. Định nghĩa mô hình PyTorch trong mô-đun `.py` độc lập

Notebook `mlp_vs_cnn` sẽ nạp lại trọng số đã huấn luyện để phân tích PCA không gian ẩn. Muốn
`torch.load` dựng lại được mô hình, định nghĩa lớp phải nằm trong một mô-đun Python thường chứ
không chỉ tồn tại trong bộ nhớ của notebook này. Ô mã dưới đây ghi trực tiếp mô-đun ra đĩa bằng
lệnh ma thuật `%%writefile`, nhờ đó mã nguồn vừa hiển thị trong báo cáo vừa nằm sẵn tại
`models/mnist_cnn_def.py`.

In [3]:
%%writefile ../models/mnist_cnn_def.py
"""Định nghĩa CNN 2D cho MNIST (Assignment 04, miền mnist).

Mô-đun độc lập để notebook khác (mlp_vs_cnn) nạp lại trọng số đã huấn luyện
tại models/mnist_cnn_pytorch.pt và trích véc-tơ ẩn 128 chiều bằng extract_features().

Tiền xử lý bắt buộc khi suy luận (xem models/mnist_preproc.json):
    x = x_uint8 / 255.0
    x = (x - mean) / std          # mean, std học từ 48000 ảnh nhánh train
    x -> tensor (N, 1, 28, 28)

Kiến trúc (theo mục 4 hợp đồng tích hợp, biến thể MNIST 2 khối 32 -> 64):
    Conv-BN-ReLU-MaxPool-Dropout x2 -> Flatten -> Dense(128) -> ReLU -> Dropout -> Dense(10)
"""

import torch
import torch.nn as nn


class MnistCNN(nn.Module):
    """CNN 2D hai khối phân cấp cho MNIST."""

    FEATURE_DIM = 128          # số chiều véc-tơ ẩn mà extract_features trả về

    def __init__(self, n_classes: int = 10, p_conv: float = 0.25, p_fc: float = 0.5):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(p_conv),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(p_conv),
        )
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, self.FEATURE_DIM)
        self.relu_fc = nn.ReLU(inplace=True)
        self.drop_fc = nn.Dropout(p_fc)
        self.fc2 = nn.Linear(self.FEATURE_DIM, n_classes)

    def extract_features(self, x: torch.Tensor) -> torch.Tensor:
        """Trả về véc-tơ ẩn 128 chiều ở tầng áp chót, dùng cho phân tích PCA.

        Tham số
        -------
        x : Tensor (N, 1, 28, 28) đã chuẩn hóa theo models/mnist_preproc.json

        Trả về
        ------
        Tensor (N, 128) sau Dense(128) và ReLU, TRƯỚC Dropout và tầng phân loại.
        """
        h = self.block1(x)
        h = self.block2(h)
        h = self.flatten(h)
        return self.relu_fc(self.fc1(h))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Trả về logit chưa qua softmax, (N, 10)."""
        return self.fc2(self.drop_fc(self.extract_features(x)))

Writing ../models/mnist_cnn_def.py


In [4]:

sys.path.insert(0, os.path.abspath(MODEL_DIR))
import mnist_cnn_def
import importlib
importlib.reload(mnist_cnn_def)
from mnist_cnn_def import MnistCNN

torch.manual_seed(RANDOM_SEED)
model_pt = MnistCNN().to(DEVICE)
n_pt = sum(p.numel() for p in model_pt.parameters() if p.requires_grad)
print(model_pt)
print()
print(f'Tổng tham số học được (PyTorch): {n_pt:,}')
for name, mod in [('block1', model_pt.block1), ('block2', model_pt.block2),
                  ('fc1', model_pt.fc1), ('fc2', model_pt.fc2)]:
    print(f'  {name:7s}: {sum(p.numel() for p in mod.parameters()):>9,} tham số')

with torch.no_grad():
    _f = model_pt.extract_features(torch.from_numpy(X_te[:4]))
print()
print('Kiểm tra extract_features: đầu vào (4, 1, 28, 28) -> đầu ra', tuple(_f.shape),
      '| kỳ vọng (4, 128):', tuple(_f.shape) == (4, 128))

MnistCNN(
  (block1): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.25, inplace=False)
  )
  (block2): Sequential(
    (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.25, inplace=False)
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (relu_fc): ReLU(inplace=True)
  (drop_fc): Dropout(p=0.5, inplace=False)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)

Tổng tham số học được (PyTorch)


**Diễn giải.** Mô hình có 421 834 tham số học được, trong đó tầng `fc1` một mình chiếm 401 536
tham số, tức $95{,}2\%$ tổng số. Đây là đặc điểm chung của mọi CNN có phần đầu phân loại kết nối
đầy đủ: hai khối tích chập chỉ tốn 18 912 tham số nhưng gánh gần như toàn bộ chi phí tính toán,
còn tầng kết nối đầy đủ thì ngược lại. Quan sát này giải thích vì sao Dropout $0{,}5$ được đặt
đúng tại đầu ra của `fc1`: nơi tập trung tham số nhiều nhất cũng là nơi dễ khớp quá nhất.

Phép kiểm tra `extract_features` xác nhận hợp đồng giao diện với notebook `mlp_vs_cnn`: đầu vào
$(4, 1, 28, 28)$ cho ra đúng $(4, 128)$.


## 5. Huấn luyện mô hình PyTorch

Bộ tối ưu Adam với $\eta = 10^{-3}$ và các siêu tham số mặc định $\beta_1 = 0{,}9$,
$\beta_2 = 0{,}999$, trùng khớp với cài đặt Adam viết tay ở notebook 01 để so sánh công bằng.
Hàm mất mát là `CrossEntropyLoss`, bản chất chính là softmax hợp nhất entropy chéo đã suy dẫn ở
mục 8.2 notebook 01. Sau mỗi epoch, trọng số được lưu lại nếu độ chính xác validation cải thiện.

In [5]:

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

train_ds = TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(y_tr))
val_ds   = TensorDataset(torch.from_numpy(X_va), torch.from_numpy(y_va))
test_ds  = TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te))

g = torch.Generator(); g.manual_seed(RANDOM_SEED)
train_ld = DataLoader(train_ds, batch_size=BATCH_FW, shuffle=True, generator=g)
val_ld   = DataLoader(val_ds,   batch_size=512, shuffle=False)
test_ld  = DataLoader(test_ds,  batch_size=512, shuffle=False)

model_pt = MnistCNN().to(DEVICE)
optimizer = torch.optim.Adam(model_pt.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()


def eval_torch(model, loader):
    model.eval()
    tot_loss, correct, n = 0.0, 0, 0
    probs = []
    with torch.no_grad():
        for xb, yb in loader:
            out = model(xb)
            tot_loss += float(criterion(out, yb)) * len(yb)
            p = torch.softmax(out, dim=1)
            probs.append(p.numpy())
            correct += int((p.argmax(1) == yb).sum())
            n += len(yb)
    return tot_loss / n, correct / n, np.concatenate(probs, axis=0)


hist_pt = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_acc_pt, best_ep_pt, best_state = -1.0, 0, None
t0 = time.time()
print('-' * 104)
for ep in range(1, EPOCHS_FW + 1):
    model_pt.train()
    run_loss, correct, n = 0.0, 0, 0
    for xb, yb in train_ld:
        optimizer.zero_grad()
        out = model_pt(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        run_loss += float(loss) * len(yb)
        correct += int((out.argmax(1) == yb).sum())
        n += len(yb)
    tr_loss, tr_acc = run_loss / n, correct / n
    va_loss, va_acc, _ = eval_torch(model_pt, val_ld)
    hist_pt['train_loss'].append(tr_loss); hist_pt['val_loss'].append(va_loss)
    hist_pt['train_acc'].append(tr_acc);   hist_pt['val_acc'].append(va_acc)
    star = ''
    if va_acc > best_acc_pt:
        best_acc_pt, best_ep_pt = va_acc, ep
        best_state = copy.deepcopy(model_pt.state_dict())
        star = '  <-- tốt nhất'
    print(f'[PyTorch] epoch {ep:2d}/{EPOCHS_FW} | train_loss={tr_loss:.4f} '
          f'train_acc={tr_acc:.4f} | val_loss={va_loss:.4f} val_acc={va_acc:.4f}{star}')
time_pt = time.time() - t0
model_pt.load_state_dict(best_state)
print('-' * 104)
print(f'[PyTorch] hoàn tất sau {time_pt:.1f}s | epoch tốt nhất = {best_ep_pt} '
      f'| val_acc tốt nhất = {best_acc_pt:.4f}')

--------------------------------------------------------------------------------------------------------


C:\Users\Nghaiz\AppData\Local\Temp\ipykernel_34940\2021512247.py:46: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  run_loss += float(loss) * len(yb)


[PyTorch] epoch  1/10 | train_loss=0.3462 train_acc=0.8914 | val_loss=0.0785 val_acc=0.9768  <-- tốt nhất


[PyTorch] epoch  2/10 | train_loss=0.1417 train_acc=0.9581 | val_loss=0.0549 val_acc=0.9837  <-- tốt nhất


[PyTorch] epoch  3/10 | train_loss=0.1110 train_acc=0.9670 | val_loss=0.0463 val_acc=0.9868  <-- tốt nhất


...

[PyTorch] epoch  6/10 | train_loss=0.0800 train_acc=0.9761 | val_loss=0.0410 val_acc=0.9868


[PyTorch] epoch  7/10 | train_loss=0.0754 train_acc=0.9769 | val_loss=0.0382 val_acc=0.9888  <-- tốt nhất


[PyTorch] epoch  8/10 | train_loss=0.0717 train_acc=0.9781 | val_loss=0.0395 val_acc=0.9885


[PyTorch] epoch  9/10 | train_loss=0.0656 train_acc=0.9789 | val_loss=0.0393 val_acc=0.9895  <-- tốt nhất


[PyTorch] epoch 10/10 | train_loss=0.0615 train_acc=0.9815 | val_loss=0.0370 val_acc=0.9893
--------------------------------------------------------------------------------------------------------
[PyTorch] hoàn tất sau 1220.8s | epoch tốt nhất = 9 | val_acc tốt nhất = 0.9895



## 6. Lưu hiện vật PyTorch cho notebook `mlp_vs_cnn`

In [6]:

ckpt_path = os.path.join(MODEL_DIR, 'mnist_cnn_pytorch.pt')
torch.save(model_pt.state_dict(), ckpt_path)
print('Đã lưu state dict :', ckpt_path, f'({os.path.getsize(ckpt_path)/1024:.1f} KB)')

# Kiểm chứng quy trình nạp lại đúng như notebook mlp_vs_cnn sẽ làm
reloaded = MnistCNN()
reloaded.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
reloaded.eval()
with torch.no_grad():
    f_new = reloaded.extract_features(torch.from_numpy(X_te[:256]))
    model_pt.eval()
    f_old = model_pt.extract_features(torch.from_numpy(X_te[:256]))
    lo_new = reloaded(torch.from_numpy(X_te[:256]))
    lo_old = model_pt(torch.from_numpy(X_te[:256]))
print('Véc-tơ ẩn sau khi nạp lại  :', tuple(f_new.shape))
print('Sai lệch tối đa đặc trưng  :', float((f_new - f_old).abs().max()))
print('Sai lệch tối đa logit      :', float((lo_new - lo_old).abs().max()))
print('Nạp lại thành công, khớp bit-for-bit:',
      bool(torch.allclose(f_new, f_old) and torch.allclose(lo_new, lo_old)))
print()
print('Ba hiện vật bàn giao cho notebook mlp_vs_cnn:')
for f in ['mnist_cnn_pytorch.pt', 'mnist_cnn_def.py', 'mnist_preproc.json']:
    p = os.path.join(MODEL_DIR, f)
    print(f'  {f:24s} {os.path.getsize(p)/1024:8.1f} KB  tồn tại={os.path.exists(p)}')

Đã lưu state dict : ../models\mnist_cnn_pytorch.pt (1654.2 KB)
Véc-tơ ẩn sau khi nạp lại  : (256, 128)
Sai lệch tối đa đặc trưng  : 0.0
Sai lệch tối đa logit      : 0.0
Nạp lại thành công, khớp bit-for-bit: True

Ba hiện vật bàn giao cho notebook mlp_vs_cnn:
  mnist_cnn_pytorch.pt       1654.2 KB  tồn tại=True
  mnist_cnn_def.py              2.4 KB  tồn tại=True
  mnist_preproc.json            0.4 KB  tồn tại=True



**Diễn giải.** Quy trình nạp lại được kiểm chứng ngay tại đây thay vì để notebook hạ nguồn phát
hiện sự cố. Sai lệch tối đa giữa véc-tơ ẩn của mô hình gốc và mô hình nạp lại bằng đúng 0, xác
nhận `state_dict` chứa đầy đủ trọng số tích chập, tham số affine $\gamma$, $\beta$ của BatchNorm
và cả hai bộ đệm trung bình trượt `running_mean`, `running_var`. Nếu thiếu bộ đệm này, mô hình
nạp lại sẽ chuẩn hóa bằng thống kê mặc định và cho kết quả sai lệch rõ rệt.

Lưu ý bắt buộc cho notebook hạ nguồn: phải gọi `model.eval()` trước khi trích đặc trưng, vì
Dropout và BatchNorm có hành vi khác nhau giữa hai chế độ.


## 7. Huấn luyện mô hình Keras

Kiến trúc được dựng lại từng tầng một cho khớp tuyệt đối với mô hình PyTorch: cùng số kênh, cùng
kích thước nhân, cùng đệm `same`, cùng thứ tự Conv-BN-ReLU-MaxPool-Dropout, cùng xác suất
Dropout, cùng bộ tối ưu Adam $\eta = 10^{-3}$.

Một khác biệt kỹ thuật cần xử lý: Keras dùng bố cục NHWC trong khi PyTorch dùng NCHW, nên dữ
liệu phải được hoán vị trục trước khi đưa vào. Đây thuần túy là quy ước bộ nhớ, không làm thay
đổi phép toán.

In [7]:

keras.utils.set_random_seed(RANDOM_SEED)

# NCHW -> NHWC
X_tr_k = np.transpose(X_tr, (0, 2, 3, 1))
X_va_k = np.transpose(X_va, (0, 2, 3, 1))
X_te_k = np.transpose(X_te, (0, 2, 3, 1))
print('Bố cục Keras:', X_tr_k.shape, '(NHWC)')

model_tf = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, 3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Conv2D(64, 3, padding='same', use_bias=False),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling2D(2),
    layers.Dropout(0.25),
    layers.Flatten(),
    layers.Dense(128),
    layers.Activation('relu'),
    layers.Dropout(0.5),
    layers.Dense(10),
], name='mnist_cnn_keras')

model_tf.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
                 loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                 metrics=['accuracy'])
model_tf.summary()
n_tf = int(sum(np.prod(w.shape) for w in model_tf.trainable_weights))
print(f'\nTổng tham số học được (Keras) : {n_tf:,}')
print(f'Tổng tham số học được (PyTorch): {n_pt:,}')
print('Hai kiến trúc khớp số tham số  :', n_tf == n_pt)

Bố cục Keras: (48000, 28, 28, 1) (NHWC)


Model: "mnist_cnn_keras"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 28, 28, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 28, 28, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 14, 14, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 14, 14, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 421,930 (1.61 MB)

 Trainable params: 421,738 (1.61 MB)

 Non-trainable params: 192 (768.00 B)


Tổng tham số học được (Keras) : 421,738
Tổng tham số học được (PyTorch): 421,738
Hai kiến trúc khớp số tham số  : True


In [8]:

class VietnameseLogger(keras.callbacks.Callback):
    '''In nhật ký từng epoch bằng tiếng Việt và ghi nhớ trọng số tốt nhất theo val_accuracy.'''
    def __init__(self, total):
        super().__init__()
        self.total = total
        self.best = -1.0
        self.best_epoch = 0
        self.best_weights = None
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        va = float(logs.get('val_accuracy', 0.0))
        star = ''
        if va > self.best:
            self.best, self.best_epoch = va, epoch + 1
            self.best_weights = self.model.get_weights()
            star = '  <-- tốt nhất'
        print(f"[Keras]   epoch {epoch+1:2d}/{self.total} | "
              f"train_loss={logs.get('loss', 0):.4f} train_acc={logs.get('accuracy', 0):.4f} | "
              f"val_loss={logs.get('val_loss', 0):.4f} val_acc={va:.4f}{star}", flush=True)

cb = VietnameseLogger(EPOCHS_FW)
print('-' * 104)
t0 = time.time()
h_tf = model_tf.fit(X_tr_k, y_tr, validation_data=(X_va_k, y_va),
                    epochs=EPOCHS_FW, batch_size=BATCH_FW, verbose=0, callbacks=[cb])
time_tf = time.time() - t0
model_tf.set_weights(cb.best_weights)
print('-' * 104)
print(f'[Keras]   hoàn tất sau {time_tf:.1f}s | epoch tốt nhất = {cb.best_epoch} '
      f'| val_acc tốt nhất = {cb.best:.4f}')

hist_tf = {
    'train_loss': [float(v) for v in h_tf.history['loss']],
    'val_loss':   [float(v) for v in h_tf.history['val_loss']],
    'train_acc':  [float(v) for v in h_tf.history['accuracy']],
    'val_acc':    [float(v) for v in h_tf.history['val_accuracy']],
}
best_ep_tf = cb.best_epoch

--------------------------------------------------------------------------------------------------------


[Keras]   epoch  1/10 | train_loss=0.5531 train_acc=0.8270 | val_loss=0.1299 val_acc=0.9649  <-- tốt nhất


[Keras]   epoch  2/10 | train_loss=0.2252 train_acc=0.9317 | val_loss=0.0663 val_acc=0.9812  <-- tốt nhất


[Keras]   epoch  3/10 | train_loss=0.1743 train_acc=0.9464 | val_loss=0.0544 val_acc=0.9857  <-- tốt nhất


[Keras]   epoch  4/10 | train_loss=0.1476 train_acc=0.9545 | val_loss=0.0548 val_acc=0.9854


...

[Keras]   epoch  7/10 | train_loss=0.1109 train_acc=0.9667 | val_loss=0.0424 val_acc=0.9887  <-- tốt nhất


[Keras]   epoch  8/10 | train_loss=0.1030 train_acc=0.9693 | val_loss=0.0416 val_acc=0.9887


[Keras]   epoch  9/10 | train_loss=0.0950 train_acc=0.9710 | val_loss=0.0397 val_acc=0.9888  <-- tốt nhất


[Keras]   epoch 10/10 | train_loss=0.0897 train_acc=0.9725 | val_loss=0.0430 val_acc=0.9883


--------------------------------------------------------------------------------------------------------
[Keras]   hoàn tất sau 126.5s | epoch tốt nhất = 9 | val_acc tốt nhất = 0.9888



## 8. Đánh giá cả hai framework trên 10 000 ảnh kiểm thử

In [9]:

def summarize(y_true, probs, loss, name):
    pred = probs.argmax(axis=1)
    acc = float((pred == y_true).mean())
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, pred, average='macro',
                                                       zero_division=0)
    cm = confusion_matrix(y_true, pred, labels=list(range(10)))
    per_class = (cm.diagonal() / cm.sum(axis=1)).astype(float)
    conf = probs.max(axis=1)
    wrong = np.where(pred != y_true)[0]
    order = wrong[np.argsort(-conf[wrong])][:8]
    hce = [{'index': int(i), 'true': int(y_true[i]), 'pred': int(pred[i]),
            'confidence': float(conf[i])} for i in order]
    print(f'--- {name} trên {len(y_true)} ảnh kiểm thử ---')
    print(f'  loss            = {loss:.6f}')
    print(f'  accuracy        = {acc:.6f}  ({acc*100:.2f}%)')
    print(f'  macro precision = {prec:.6f}')
    print(f'  macro recall    = {rec:.6f}')
    print(f'  macro F1        = {f1:.6f}')
    print(f'  số ảnh sai      = {len(wrong)}')
    return dict(loss=float(loss), accuracy=acc, macro_precision=float(prec),
                macro_recall=float(rec), macro_f1=float(f1),
                confusion_matrix=cm.tolist(), per_class_accuracy=per_class.tolist(),
                high_conf_errors=hce, pred=pred, probs=probs)

loss_pt_te, acc_pt_te, probs_pt = eval_torch(model_pt, test_ld)
res_pt = summarize(y_te, probs_pt, loss_pt_te, 'PyTorch')
print()

logits_tf = model_tf.predict(X_te_k, batch_size=512, verbose=0)
probs_tf = tf.nn.softmax(logits_tf).numpy()
loss_tf_te = float(keras.losses.SparseCategoricalCrossentropy(from_logits=True)(
    y_te, logits_tf).numpy())
res_tf = summarize(y_te, probs_tf, loss_tf_te, 'Keras / TensorFlow')

print()
print('Chênh lệch PyTorch so với Keras:')
print(f'  accuracy : {(res_pt["accuracy"]-res_tf["accuracy"])*100:+.2f} điểm phần trăm')
print(f'  macro F1 : {(res_pt["macro_f1"]-res_tf["macro_f1"])*100:+.2f} điểm phần trăm')
print(f'  thời gian: {time_pt:.1f}s so với {time_tf:.1f}s '
      f'(tỉ lệ {time_tf/max(time_pt,1e-9):.2f}x)')

--- PyTorch trên 10000 ảnh kiểm thử ---
  loss            = 0.030118
  accuracy        = 0.989800  (98.98%)
  macro precision = 0.989867
  macro recall    = 0.989733
  macro F1        = 0.989777
  số ảnh sai      = 102



--- Keras / TensorFlow trên 10000 ảnh kiểm thử ---
  loss            = 0.030736
  accuracy        = 0.990200  (99.02%)
  macro precision = 0.990077
  macro recall    = 0.990241
  macro F1        = 0.990146
  số ảnh sai      = 98

Chênh lệch PyTorch so với Keras:
  accuracy : -0.04 điểm phần trăm
  macro F1 : -0.04 điểm phần trăm
  thời gian: 1220.8s so với 126.5s (tỉ lệ 0.10x)


In [10]:

best_name = 'PyTorch' if res_pt['accuracy'] >= res_tf['accuracy'] else 'Keras / TensorFlow'
res_best  = res_pt if res_pt['accuracy'] >= res_tf['accuracy'] else res_tf
print(f'Mô hình tốt nhất: {best_name}')
print()
print(f'Báo cáo phân loại chi tiết của {best_name} trên 10 000 ảnh kiểm thử:')
print(classification_report(y_te, res_best['pred'], digits=4, zero_division=0))

Mô hình tốt nhất: Keras / TensorFlow

Báo cáo phân loại chi tiết của Keras / TensorFlow trên 10 000 ảnh kiểm thử:
              precision    recall  f1-score   support

           0     0.9959    0.9908    0.9934       980
           1     0.9956    0.9956    0.9956      1135
           2     0.9865    0.9893    0.9879      1032
           3     0.9950    0.9901    0.9926      1010
           4     0.9949    0.9929    0.9939       982
           5     0.9802    0.9978    0.9889       892
           6     0.9937    0.9896    0.9916       958
           7     0.9883    0.9844    0.9864      1028
           8     0.9827    0.9908    0.9867       974
           9     0.9880    0.9812    0.9846      1009

    accuracy                         0.9902     10000
   macro avg     0.9901    0.9902    0.9901     10000
weighted avg     0.9902    0.9902    0.9902     10000




## 9. Hình 1: đường cong huấn luyện PyTorch so với Keras

In [11]:

ep_axis = np.arange(1, EPOCHS_FW + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ep_axis, hist_pt['train_loss'], 'o-',  color='#8E44AD', label='PyTorch train')
axes[0].plot(ep_axis, hist_pt['val_loss'],  'o--', color='#8E44AD', alpha=0.55, label='PyTorch val')
axes[0].plot(ep_axis, hist_tf['train_loss'], 's-',  color='#D35400', label='Keras train')
axes[0].plot(ep_axis, hist_tf['val_loss'],  's--', color='#D35400', alpha=0.55, label='Keras val')
axes[0].set_title('Mất mát entropy chéo theo epoch', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Mất mát'); axes[0].set_yscale('log')
axes[0].legend(fontsize=9); axes[0].set_xticks(ep_axis)

axes[1].plot(ep_axis, np.array(hist_pt['train_acc'])*100, 'o-',  color='#8E44AD', label='PyTorch train')
axes[1].plot(ep_axis, np.array(hist_pt['val_acc'])*100,  'o--', color='#8E44AD', alpha=0.55, label='PyTorch val')
axes[1].plot(ep_axis, np.array(hist_tf['train_acc'])*100, 's-',  color='#D35400', label='Keras train')
axes[1].plot(ep_axis, np.array(hist_tf['val_acc'])*100,  's--', color='#D35400', alpha=0.55, label='Keras val')
axes[1].axvline(best_ep_pt, color='#8E44AD', ls=':', lw=1.2)
axes[1].axvline(best_ep_tf, color='#D35400', ls=':', lw=1.2)
axes[1].set_title('Độ chính xác theo epoch (đường chấm dọc = epoch tốt nhất)', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Độ chính xác (%)')
axes[1].legend(fontsize=9, loc='lower right'); axes[1].set_xticks(ep_axis)

fig.suptitle('CNN sâu trên MNIST: PyTorch so với Keras\n'
             f'(huấn luyện đầy đủ {len(X_tr)} ảnh train, {len(X_va)} ảnh validation, batch {BATCH_FW})',
             fontsize=14, y=1.04)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mnist_framework_curves.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_framework_curves.png')
print(f'val_acc cuối : PyTorch {hist_pt["val_acc"][-1]*100:.2f}% | Keras {hist_tf["val_acc"][-1]*100:.2f}%')
print(f'val_acc tốt nhất: PyTorch {max(hist_pt["val_acc"])*100:.2f}% (epoch {best_ep_pt}) | '
      f'Keras {max(hist_tf["val_acc"])*100:.2f}% (epoch {best_ep_tf})')
print(f'train_acc cuối : PyTorch {hist_pt["train_acc"][-1]*100:.2f}% | Keras {hist_tf["train_acc"][-1]*100:.2f}%')

Đã lưu fig_mnist_framework_curves.png
val_acc cuối : PyTorch 98.93% | Keras 98.83%
val_acc tốt nhất: PyTorch 98.95% (epoch 9) | Keras 98.88% (epoch 9)
train_acc cuối : PyTorch 98.15% | Keras 97.25%


<!-- INTERP_FWCURVES -->


## 10. Hình 2: ma trận nhầm lẫn của hai framework

In [12]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
for ax, res, name in [(axes[0], res_pt, 'PyTorch'), (axes[1], res_tf, 'Keras / TensorFlow')]:
    cm = np.array(res['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', ax=ax, cbar=False,
                annot_kws={'size': 8}, linewidths=0.4, linecolor='#DDDDDD')
    ax.set_title(f'{name}: accuracy = {res["accuracy"]*100:.2f}%, '
                 f'{int(cm.sum() - np.trace(cm))} ảnh sai', fontsize=12)
    ax.set_xlabel('Nhãn dự đoán'); ax.set_ylabel('Nhãn thật')
fig.suptitle('Ma trận nhầm lẫn trên 10 000 ảnh kiểm thử, CNN sâu hai framework',
             fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mnist_framework_confusion.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_framework_confusion.png')

for tag, res in [('PyTorch', res_pt), ('Keras', res_tf)]:
    cm = np.array(res['confusion_matrix'])
    off = sorted([(cm[a, b], a, b) for a in range(10) for b in range(10)
                  if a != b and cm[a, b] > 0], reverse=True)[:5]
    print(f'\nNăm cặp nhầm lẫn nặng nhất của {tag}:')
    for n, a, b in off:
        print(f'  {a} -> {b} : {n} ảnh ({100*n/cm[a].sum():.2f}% số ảnh lớp {a})')

Đã lưu fig_mnist_framework_confusion.png

Năm cặp nhầm lẫn nặng nhất của PyTorch:
  2 -> 7 : 8 ảnh (0.78% số ảnh lớp 2)
  4 -> 9 : 7 ảnh (0.71% số ảnh lớp 4)
  6 -> 0 : 6 ảnh (0.63% số ảnh lớp 6)
  9 -> 7 : 5 ảnh (0.50% số ảnh lớp 9)
  9 -> 4 : 5 ảnh (0.50% số ảnh lớp 9)

Năm cặp nhầm lẫn nặng nhất của Keras:
  7 -> 2 : 7 ảnh (0.68% số ảnh lớp 7)
  3 -> 5 : 7 ảnh (0.69% số ảnh lớp 3)
  2 -> 7 : 6 ảnh (0.58% số ảnh lớp 2)
  9 -> 8 : 5 ảnh (0.50% số ảnh lớp 9)
  9 -> 5 : 5 ảnh (0.50% số ảnh lớp 9)


<!-- INTERP_FWCFM -->


## 11. Hình 3: đối chiếu ba cách cài đặt

Kết quả của hai mô hình NumPy được nạp lại từ tệp trung gian mà notebook 01 đã ghi. Cần nhắc lại
một lần nữa để tránh đọc sai: hai mô hình NumPy huấn luyện trên **tập con phân tầng 12 000 ảnh**
còn hai mô hình framework huấn luyện trên **toàn bộ 48 000 ảnh**. Cả bốn mô hình đều được đánh
giá trên cùng 10 000 ảnh kiểm thử, nên phép so sánh công bằng ở phía đánh giá nhưng **không**
công bằng ở phía dữ liệu huấn luyện. Chênh lệch quan sát được vì vậy là tổng hợp của hai yếu tố:
sức mạnh kiến trúc và lượng dữ liệu.

In [13]:

with open(os.path.join(REP_DIR, 'metrics_mnist_scratch_partial.json'), encoding='utf-8') as f:
    partial = json.load(f)
np_base, np_impr = partial['numpy_baseline'], partial['numpy_improved']
sub = partial['subset']
EPOCHS_NP = int(sub['epochs'])
print('Nạp lại kết quả NumPy từ notebook 01:')
print(f"  numpy_baseline : acc={np_base['accuracy']*100:.2f}% macroF1={np_base['macro_f1']*100:.2f}% "
      f"params={np_base['params']:,} time={np_base['train_time_s']:.1f}s")
print(f"  numpy_improved : acc={np_impr['accuracy']*100:.2f}% macroF1={np_impr['macro_f1']*100:.2f}% "
      f"params={np_impr['params']:,} time={np_impr['train_time_s']:.1f}s")
print(f"  tập con huấn luyện NumPy: {sub['n_train_subset']} train / {sub['n_val_subset']} val")
print(f"  kiểm tra gradient: {partial['gradient_check']['n_checks']} phép, "
      f"sai số tương đối lớn nhất = {partial['gradient_check']['max_rel_error']:.3e}")

Nạp lại kết quả NumPy từ notebook 01:
  numpy_baseline : acc=97.03% macroF1=97.01% params=27,562 time=120.0s
  numpy_improved : acc=98.23% macroF1=98.22% params=52,138 time=148.0s
  tập con huấn luyện NumPy: 12000 train / 3000 val
  kiểm tra gradient: 16 phép, sai số tương đối lớn nhất = 1.861e-10


In [14]:

names   = ['NumPy (Improved)', 'PyTorch', 'Keras / TensorFlow']
metrics = ['Accuracy', 'Macro Precision', 'Macro Recall', 'Macro F1']
table = np.array([
    [np_impr['accuracy'], np_impr['macro_precision'], np_impr['macro_recall'], np_impr['macro_f1']],
    [res_pt['accuracy'],  res_pt['macro_precision'],  res_pt['macro_recall'],  res_pt['macro_f1']],
    [res_tf['accuracy'],  res_tf['macro_precision'],  res_tf['macro_recall'],  res_tf['macro_f1']],
]) * 100

xpos = np.arange(len(metrics)); w = 0.26
colors = ['#C0392B', '#8E44AD', '#D35400']
fig, ax = plt.subplots(figsize=(12.5, 6.2))
for i, nm in enumerate(names):
    bars = ax.bar(xpos + (i - 1) * w, table[i], w, label=nm,
                  color=colors[i], edgecolor='black', linewidth=0.6)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.06,
                f'{bar.get_height():.2f}', ha='center', fontsize=8.5)
ax.set_xticks(xpos); ax.set_xticklabels(metrics)
ax.set_ylabel('Giá trị (%)')
ax.set_ylim(table.min() - 1.2, 100.4)
ax.set_title('MNIST: đối chiếu ba cách cài đặt CNN 2D trên 10 000 ảnh kiểm thử\n'
             f'(NumPy huấn luyện trên tập con {sub["n_train_subset"]} ảnh; '
             f'PyTorch và Keras trên đủ {len(X_tr)} ảnh)', fontsize=13)
ax.legend(fontsize=9.5, loc='lower right')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mnist_3way_benchmark.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_3way_benchmark.png')
print()
hdr = f"{'Cách cài đặt':<20}{'Accuracy':>11}{'MacroP':>10}{'MacroR':>10}{'MacroF1':>10}{'Tham số':>12}{'Thời gian':>12}{'Ảnh train':>11}"
print(hdr); print('-' * len(hdr))
rows_meta = [(names[0], np_impr['params'], np_impr['train_time_s'], sub['n_train_subset']),
             (names[1], n_pt, time_pt, len(X_tr)),
             (names[2], n_tf, time_tf, len(X_tr))]
for i, (nm, pr, tt, ntr) in enumerate(rows_meta):
    print(f'{nm:<20}{table[i,0]:>10.2f}%{table[i,1]:>9.2f}%{table[i,2]:>9.2f}%'
          f'{table[i,3]:>9.2f}%{pr:>12,}{tt:>11.1f}s{ntr:>11,}')
print('-' * len(hdr))
print(f'Baseline NumPy (tham chiếu): accuracy {np_base["accuracy"]*100:.2f}%, '
      f'macro F1 {np_base["macro_f1"]*100:.2f}%, {np_base["params"]:,} tham số')

Đã lưu fig_mnist_3way_benchmark.png

Cách cài đặt           Accuracy    MacroP    MacroR   MacroF1     Tham số   Thời gian  Ảnh train
------------------------------------------------------------------------------------------------
NumPy (Improved)         98.23%    98.23%    98.21%    98.22%      52,138      148.0s     12,000
PyTorch                  98.98%    98.99%    98.97%    98.98%     421,738     1220.8s     48,000
Keras / TensorFlow       99.02%    99.01%    99.02%    99.01%     421,738      126.5s     48,000
------------------------------------------------------------------------------------------------
Baseline NumPy (tham chiếu): accuracy 97.03%, macro F1 97.01%, 27,562 tham số


<!-- INTERP_3WAY -->


## 12. Hình 4: độ chính xác theo từng lớp của mô hình tốt nhất

In [15]:

pca_ = np.array(res_best['per_class_accuracy']) * 100
support = np.bincount(y_te, minlength=10)
order_worst = np.argsort(pca_)

fig, ax = plt.subplots(figsize=(12, 6))
cols = ['#27AE60' if v >= pca_.mean() else '#E74C3C' for v in pca_]
bars = ax.bar(np.arange(10), pca_, color=cols, edgecolor='black', linewidth=0.6)
ax.axhline(pca_.mean(), color='navy', ls='--', lw=1.4,
           label=f'Trung bình theo lớp = {pca_.mean():.2f}%')
ax.axhline(res_best['accuracy']*100, color='darkorange', ls=':', lw=1.6,
           label=f'Độ chính xác tổng thể = {res_best["accuracy"]*100:.2f}%')
for k, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.04,
            f'{pca_[k]:.2f}', ha='center', fontsize=9)
ax.set_xticks(np.arange(10)); ax.set_xlabel('Chữ số'); ax.set_ylabel('Độ chính xác (%)')
ax.set_ylim(max(0, pca_.min() - 1.2), 100.35)
for k, bar in enumerate(bars):
    ax.text(bar.get_x() + bar.get_width()/2, ax.get_ylim()[0] + 0.12,
            f'n={support[k]}', ha='center', va='bottom', fontsize=8, color='white')
ax.set_title(f'Độ chính xác theo từng lớp của mô hình tốt nhất ({best_name})\n'
             'trên 10 000 ảnh kiểm thử MNIST', fontsize=13)
ax.legend(fontsize=9.5, loc='lower right')
fig.tight_layout()
fig.savefig(f'{FIG_DIR}/fig_mnist_per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_per_class_accuracy.png')
print()
print(f"{'Chữ số':>7}{'Accuracy':>11}{'Số ảnh':>9}{'Số ảnh sai':>13}")
print('-' * 40)
cmb = np.array(res_best['confusion_matrix'])
for k in range(10):
    print(f'{k:>7}{pca_[k]:>10.2f}%{support[k]:>9}{int(support[k]-cmb[k,k]):>13}')
print('-' * 40)
print(f'Lớp tốt nhất : chữ số {int(pca_.argmax())} với {pca_.max():.2f}%')
print(f'Lớp kém nhất : chữ số {int(pca_.argmin())} với {pca_.min():.2f}%')
print(f'Biên độ dao động giữa lớp tốt nhất và kém nhất: {pca_.max()-pca_.min():.2f} điểm phần trăm')
print('Ba lớp khó nhất theo thứ tự:',
      ', '.join(f'{int(k)} ({pca_[k]:.2f}%)' for k in order_worst[:3]))

Đã lưu fig_mnist_per_class_accuracy.png

 Chữ số   Accuracy   Số ảnh   Số ảnh sai
----------------------------------------
      0     99.08%      980            9
      1     99.56%     1135            5
      2     98.93%     1032           11
      3     99.01%     1010           10
      4     99.29%      982            7
      5     99.78%      892            2
      6     98.96%      958           10
      7     98.44%     1028           16
      8     99.08%      974            9
      9     98.12%     1009           19
----------------------------------------
Lớp tốt nhất : chữ số 5 với 99.78%
Lớp kém nhất : chữ số 9 với 98.12%
Biên độ dao động giữa lớp tốt nhất và kém nhất: 1.66 điểm phần trăm
Ba lớp khó nhất theo thứ tự: 9 (98.12%), 7 (98.44%), 2 (98.93%)


<!-- INTERP_PERCLASS -->


## 13. Hình 5: tám ảnh sai với độ tin cậy cao nhất

Sai lầm tự tin là loại sai nguy hiểm nhất trong ứng dụng thực tế, vì cơ chế lọc theo ngưỡng tin
cậy không bắt được chúng. Ta trực quan hóa tám ảnh mà mô hình tốt nhất dự đoán sai nhưng lại gán
xác suất cao nhất.

In [16]:

hce = res_best['high_conf_errors']
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, item in zip(axes.ravel(), hce):
    i = item['index']
    ax.imshow(x_test_raw[i], cmap='gray_r', vmin=0, vmax=255)
    ax.set_title(f"#{i} · thật = {item['true']} · dự đoán = {item['pred']}\n"
                 f"độ tin cậy = {item['confidence']*100:.2f}%", fontsize=10.5, color='#B03A2E')
    ax.set_xticks([]); ax.set_yticks([])
for ax in axes.ravel()[len(hce):]:
    ax.axis('off')
fig.suptitle(f'Tám ảnh bị phân loại sai với độ tin cậy cao nhất ({best_name})\n'
             'trên 10 000 ảnh kiểm thử MNIST', fontsize=14, y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(f'{FIG_DIR}/fig_mnist_high_conf_errors.png', dpi=150, bbox_inches='tight')
plt.close(fig)
print('Đã lưu fig_mnist_high_conf_errors.png')
print()
print(f"{'STT':>4}{'Chỉ số ảnh':>12}{'Nhãn thật':>11}{'Dự đoán':>9}{'Độ tin cậy':>13}")
print('-' * 50)
for r, item in enumerate(hce, 1):
    print(f"{r:>4}{item['index']:>12}{item['true']:>11}{item['pred']:>9}"
          f"{item['confidence']*100:>12.2f}%")
print('-' * 50)
conf_all = res_best['probs'].max(axis=1)
pred_all = res_best['pred']
wrong_mask = pred_all != y_te
print(f'Độ tin cậy trung bình khi dự đoán ĐÚNG : {conf_all[~wrong_mask].mean()*100:.2f}%')
print(f'Độ tin cậy trung bình khi dự đoán SAI  : {conf_all[wrong_mask].mean()*100:.2f}%')
print(f'Số ảnh sai có độ tin cậy > 99%         : {int(((conf_all > 0.99) & wrong_mask).sum())}')
print(f'Tổng số ảnh sai                        : {int(wrong_mask.sum())}')

Đã lưu fig_mnist_high_conf_errors.png

 STT  Chỉ số ảnh  Nhãn thật  Dự đoán   Độ tin cậy
--------------------------------------------------
   1        2654          6        1       99.98%
   2        2043          4        8       99.87%
   3        9729          5        6       99.82%
   4        1014          6        5       99.69%
   5        3520          6        4       99.67%
   6        2118          6        0       99.62%
   7        9015          7        2       99.55%
   8        3073          1        2       99.54%
--------------------------------------------------
Độ tin cậy trung bình khi dự đoán ĐÚNG : 99.38%
Độ tin cậy trung bình khi dự đoán SAI  : 71.62%
Số ảnh sai có độ tin cậy > 99%         : 9
Tổng số ảnh sai                        : 98


<!-- INTERP_HCE -->


## 14. Lắp ráp tệp `reports/metrics_mnist.json`

Tệp tổng hợp tuân thủ schema mục 5.3 hợp đồng, biến thể phân loại mười lớp: bốn khóa mô hình
`numpy_baseline`, `numpy_improved`, `pytorch`, `tensorflow`, mỗi khóa mang `macro_precision`,
`macro_recall`, `macro_f1`, ma trận nhầm lẫn $10 \times 10$, mảng `per_class_accuracy` mười phần
tử và danh sách `high_conf_errors` tối đa tám phần tử. Mọi tỉ lệ ghi ở thang $[0, 1]$ với đầy đủ
chữ số thập phân, không làm tròn thành chuỗi.

In [17]:

def pack_fw(res, hist, best_ep, ttime, n_params, framework, epochs):
    return {
        'framework': framework,
        'params': int(n_params),
        'train_time_s': float(ttime),
        'epochs': int(epochs),
        'best_epoch': int(best_ep),
        'accuracy': res['accuracy'],
        'macro_precision': res['macro_precision'],
        'macro_recall': res['macro_recall'],
        'macro_f1': res['macro_f1'],
        'loss': res['loss'],
        'history': {k: [float(v) for v in hist[k]]
                    for k in ('train_loss', 'val_loss', 'train_acc', 'val_acc')},
        'confusion_matrix': res['confusion_matrix'],
        'per_class_accuracy': res['per_class_accuracy'],
        'high_conf_errors': res['high_conf_errors'],
    }

notes = (
    'Hai mô hình NumPy thuần (numpy_baseline, numpy_improved) được huấn luyện trên TẬP CON '
    f'PHÂN TẦNG {sub["n_train_subset"]} ảnh train và {sub["n_val_subset"]} ảnh validation '
    f'({EPOCHS_NP} epoch, batch {sub["batch_size"]}) thay vì toàn bộ 48000/12000, do mạng tích '
    'chập cài bằng NumPy thuần chạy trên CPU vượt ngân sách 15 phút mỗi notebook mà hợp đồng quy '
    'định. Hai mô hình framework (pytorch, tensorflow) dùng ĐẦY ĐỦ 48000 ảnh train và 12000 ảnh '
    f'validation, {EPOCHS_FW} epoch, batch {BATCH_FW}. CẢ BỐN mô hình đều được đánh giá trên trọn '
    'vẹn 10000 ảnh của tập kiểm thử gốc nên phép so sánh công bằng ở phía đánh giá; chênh lệch '
    'giữa nhóm NumPy và nhóm framework là tổng hợp của hai yếu tố kiến trúc và lượng dữ liệu '
    'huấn luyện. Khóa roc_auc của schema nhị phân không áp dụng cho bài toán 10 lớp nên được lược '
    'bỏ, thay bằng macro_precision / macro_recall / macro_f1 / per_class_accuracy / '
    'high_conf_errors đúng theo biến thể đa lớp của mục 5.3. Kiểm tra gradient bằng sai phân hữu '
    f'hạn trung tâm (eps=1e-5, float64) trên {partial["gradient_check"]["n_checks"]} vị trí tham '
    f'số cho sai số tương đối lớn nhất {partial["gradient_check"]["max_rel_error"]:.3e}, xác nhận '
    'phép lan truyền ngược viết tay là đúng. Notebook 01 còn ghi tệp trung gian '
    'reports/metrics_mnist_scratch_partial.json chứa nhật ký chi tiết của kiểm tra gradient; tệp '
    'đó bổ sung chứ không thay thế tệp bắt buộc này.'
)

metrics = {
    'domain': 'mnist',
    'task': 'classification',
    'dataset': {
        'file': 'mnist/data/mnist.npz',
        'n_raw': int(len(x_train_raw) + len(x_test_raw)),
        'n_clean': int(len(x_train_raw) + len(x_test_raw)),
        'n_train': int(len(X_tr)),
        'n_val': int(len(X_va)),
        'n_test': int(len(X_te)),
        'n_features': 784,
        'image_shape': [28, 28, 1],
        'n_classes': 10,
        'preprocess': {'mean': MEAN, 'std': STD, 'scale': 255.0},
        'numpy_subset': {'n_train': int(sub['n_train_subset']),
                         'n_val': int(sub['n_val_subset'])},
    },
    'models': {
        'numpy_baseline': {k: v for k, v in np_base.items()},
        'numpy_improved': {k: v for k, v in np_impr.items()},
        'pytorch':    pack_fw(res_pt, hist_pt, best_ep_pt, time_pt, n_pt,
                              'PyTorch 2.9.1 (CPU)', EPOCHS_FW),
        'tensorflow': pack_fw(res_tf, hist_tf, best_ep_tf, time_tf, n_tf,
                              'TensorFlow 2.21.0 / Keras 3.15.1 (CPU)', EPOCHS_FW),
    },
    'best_model': 'pytorch' if res_pt['accuracy'] >= res_tf['accuracy'] else 'tensorflow',
    'gradient_check': partial['gradient_check'],
    'notes': notes,
}

out = os.path.join(REP_DIR, 'metrics_mnist.json')
with open(out, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)
print('Đã ghi', out, f'({os.path.getsize(out)/1024:.1f} KB)')
print()
for key, m in metrics['models'].items():
    assert len(m['confusion_matrix']) == 10 and len(m['confusion_matrix'][0]) == 10
    assert len(m['per_class_accuracy']) == 10
    assert len(m['high_conf_errors']) <= 8
    print(f"  {key:16s} acc={m['accuracy']:.6f} macroF1={m['macro_f1']:.6f} "
          f"params={m['params']:>9,} time={m['train_time_s']:8.1f}s "
          f"cfm=10x10 per_class=10 hce={len(m['high_conf_errors'])}")
print()
print('Mô hình tốt nhất ghi trong metrics:', metrics['best_model'])

Đã ghi ../reports\metrics_mnist.json (25.7 KB)

  numpy_baseline   acc=0.970300 macroF1=0.970091 params=   27,562 time=   120.0s cfm=10x10 per_class=10 hce=8
  numpy_improved   acc=0.982300 macroF1=0.982176 params=   52,138 time=   148.0s cfm=10x10 per_class=10 hce=8
  pytorch          acc=0.989800 macroF1=0.989777 params=  421,738 time=  1220.8s cfm=10x10 per_class=10 hce=8
  tensorflow       acc=0.990200 macroF1=0.990146 params=  421,738 time=   126.5s cfm=10x10 per_class=10 hce=8

Mô hình tốt nhất ghi trong metrics: tensorflow


In [18]:

required = [
    'fig_mnist_class_distribution.png', 'fig_mnist_sample_grid.png',
    'fig_mnist_scratch_curves.png', 'fig_mnist_scratch_confusion.png',
    'fig_mnist_scratch_comparison.png', 'fig_mnist_framework_curves.png',
    'fig_mnist_framework_confusion.png', 'fig_mnist_3way_benchmark.png',
    'fig_mnist_per_class_accuracy.png', 'fig_mnist_high_conf_errors.png',
]
print('Kiểm tra đủ 10 hình bắt buộc theo mục 6 hợp đồng tích hợp:')
ok = True
for f in required:
    p = os.path.join(FIG_DIR, f)
    e = os.path.exists(p)
    ok &= e
    print(f"  {'OK ' if e else 'THIẾU'} {f:40s} {os.path.getsize(p)/1024 if e else 0:8.1f} KB")
print()
print('Đủ 10/10 hình:', ok)
print()
print('Hiện vật mô hình:')
for f in ['mnist_cnn_pytorch.pt', 'mnist_cnn_def.py', 'mnist_preproc.json']:
    p = os.path.join(MODEL_DIR, f)
    print(f'  {f:24s} {os.path.getsize(p)/1024:8.1f} KB')

Kiểm tra đủ 10 hình bắt buộc theo mục 6 hợp đồng tích hợp:
  OK  fig_mnist_class_distribution.png             85.2 KB
  OK  fig_mnist_sample_grid.png                    97.6 KB
  OK  fig_mnist_scratch_curves.png                161.2 KB
  OK  fig_mnist_scratch_confusion.png              87.1 KB
  OK  fig_mnist_scratch_comparison.png             64.4 KB
  OK  fig_mnist_framework_curves.png              161.8 KB
  OK  fig_mnist_framework_confusion.png            80.4 KB
  OK  fig_mnist_3way_benchmark.png                 64.5 KB
  OK  fig_mnist_per_class_accuracy.png            228.1 KB
  OK  fig_mnist_high_conf_errors.png               74.2 KB

Đủ 10/10 hình: True

Hiện vật mô hình:
  mnist_cnn_pytorch.pt       1654.2 KB
  mnist_cnn_def.py              2.4 KB
  mnist_preproc.json            0.4 KB


<!-- INTERP_FINAL2 -->